# Portofolio Data Science - Pertemuan 9
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

---

## Langkah 1: Memuat dan Mengeksplorasi Dataset Kanker Payudara

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Memuat dataset kanker payudara
kanker_data = load_breast_cancer()
df_fitur = pd.DataFrame(kanker_data.data, columns=kanker_data.feature_names)
target = kanker_data.target # 0 = malignant (ganas), 1 = benign (jinak)

print("Dimensi Dataset Fitur:", df_fitur.shape)
print("Proporsi Target (0: Malignant, 1: Benign):")
print(pd.Series(target).value_counts(normalize=True).round(3))

## Langkah 2: Pembagian Dataset dan Standardisasi Fitur

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data dengan rasio 80:20 secara stratified
X_train, X_test, y_train, y_test = train_test_split(
    df_fitur, target, test_size=0.2, random_state=42, stratify=target
)

# Penskalaan fitur menggunakan StandardScaler
scaler_std = StandardScaler()
X_train_scaled = scaler_std.fit_transform(X_train)
X_test_scaled = scaler_std.transform(X_test)

print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])

## Langkah 3: Pemodelan Menggunakan Regresi Logistik

In [ ]:
from sklearn.linear_model import LogisticRegression

# Inisialisasi dan training model Logistic Regression
lr_model = LogisticRegression(max_iter=5000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Prediksi ulasan target data testing
y_pred_lr = lr_model.predict(X_test_scaled)

# Menampilkan koefisien fitur yang paling mempengaruhi klasifikasi
df_koefisien = pd.DataFrame({
    'Fitur': df_fitur.columns,
    'Nilai Koefisien': lr_model.coef_[0]
}).sort_values('Nilai Koefisien', key=abs, ascending=False)

print("5 Fitur Terpenting berdasarkan Koefisien Absolut:")
print(df_koefisien.head())

## Langkah 4: Pemodelan Menggunakan Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Training model Decision Tree dengan max_depth=4
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

# Prediksi target data testing
y_pred_dt = dt_model.predict(X_test)

# Visualisasi diagram Decision Tree
plt.figure(figsize=(18, 9))
plot_tree(
    dt_model,
    feature_names=df_fitur.columns,
    class_names=['Malignant', 'Benign'],
    filled=True,
    rounded=True
)
plt.title("Visualisasi Struktur Pohon Keputusan (Decision Tree)", fontsize=15)
plt.show()

## Langkah 5: Evaluasi dan Perbandingan Performa Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("=== HASIL EVALUASI REGRESI LOGISTIK ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred_lr):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Malignant', 'Benign']))

print("\n" + "="*45 + "\n")

print("=== HASIL EVALUASI DECISION TREE ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred_dt):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['Malignant', 'Benign']))

## Kesimpulan & Pembahasan

Berdasarkan analisis klasifikasi yang telah dilakukan terhadap Breast Cancer dataset:
1. **Perbandingan Performa**: Model Regresi Logistik memberikan tingkat akurasi yang lebih tinggi dan stabil dibandingkan dengan Decision Tree. Hal ini menunjukkan bahwa dataset klasifikasi kanker ini cenderung memiliki batas keputusan linear yang optimal setelah penskalaan fitur dilakukan.
2. **Pentingnya Fitur**: Fitur dengan nilai koefisien absolut tinggi pada Regresi Logistik sangat menentukan probabilitas tumor dikategorikan sebagai ganas. Sementara itu, Decision Tree mempermudah visualisasi proses pengambilan keputusan secara bertahap.
3. **Evaluasi Klinis**: Pada kasus diagnosis penyakit kanker payudara, performa model yang meminimalkan *false negative* (pasien kanker terdiagnosis normal) sangat krusial, di mana Regresi Logistik menunjukkan hasil recall yang sangat tinggi untuk kelas Malignant.